In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [4]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [5]:
# Q1 label encoding
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train["label"] = train["answer"].map(label_map)

print(train.loc[150, "label"])

2


In [7]:
#Q2
formatted = str(train.loc[0, "prompt"]) + " [SEP] " + str(train.loc[0, "B"])

print(len(formatted))

407


In [9]:
#Q3
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row = train.loc[0]

choices = [
    row["A"],
    row["B"],
    row["C"],
    row["D"],
    row["E"]
]

formatted = [row["prompt"] + " [SEP] " + c for c in choices]

encoding = tokenizer(
    formatted,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

encoding = {k: v.unsqueeze(0) for k, v in encoding.items()}

print(encoding["input_ids"].shape)
print(encoding["input_ids"].shape[1])

torch.Size([1, 5, 128])
5


In [12]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [13]:
# Q7 Lora

from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, config)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(trainable)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


295681


In [14]:
#Q10
import torch

row = train.loc[0]

choices = [
    row["A"],
    row["B"],
    row["C"],
    row["D"],
    row["E"]
]

formatted = [row["prompt"] + " [SEP] " + c for c in choices]

encoding = tokenizer(
    formatted,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

encoding = {k: v.unsqueeze(0) for k, v in encoding.items()}

model.eval()

with torch.no_grad():
    outputs = model(**encoding)

probabilities = torch.softmax(outputs.logits, dim=1)

print(probabilities)

print(round(probabilities[0, 4].item(), 4))

tensor([[0.1975, 0.2001, 0.2030, 0.1992, 0.2003]])
0.2003
